## ConditionalVAE 条件变分自编码器测试

In [10]:
import torch
import unittest
from models import CVAE

from tools import register_hooks
class TestCVAE(unittest.TestCase):

    def setUp(self) -> None:
        # self.model2 = VAE(3, 10)
        self.model = CVAE(3, 40, 10)

    def test_forward(self):
        hooks = register_hooks(self.model)
        x = torch.randn(16, 3, 64, 64)
        c = torch.randn(16, 40)
        y = self.model(x, labels = c)
        print("Model Output size:", y[0].size())
        # print("Model2 Output size:", self.model2(x)[0].size())

    def test_loss(self):
        x = torch.randn(16, 3, 64, 64)
        c = torch.randn(16, 40)
        result = self.model(x, labels = c)
        loss = self.model.loss_function(*result, M_N = 0.005)
        print(loss)

unittest.main(defaultTest='TestCVAE', argv=['first-arg-is-ignored'], exit=False)

..
----------------------------------------------------------------------
Ran 2 tests in 0.167s

OK


[embed_class] -> input[0]: [16, 40] | output: [16, 4096]
[embed_data] -> input[0]: [16, 3, 64, 64] | output: [16, 3, 64, 64]
[encoder.0.0] -> input[0]: [16, 4, 64, 64] | output: [16, 32, 32, 32]
[encoder.0.1] -> input[0]: [16, 32, 32, 32] | output: [16, 32, 32, 32]
[encoder.0.2] -> input[0]: [16, 32, 32, 32] | output: [16, 32, 32, 32]
[encoder.1.0] -> input[0]: [16, 32, 32, 32] | output: [16, 64, 16, 16]
[encoder.1.1] -> input[0]: [16, 64, 16, 16] | output: [16, 64, 16, 16]
[encoder.1.2] -> input[0]: [16, 64, 16, 16] | output: [16, 64, 16, 16]
[encoder.2.0] -> input[0]: [16, 64, 16, 16] | output: [16, 128, 8, 8]
[encoder.2.1] -> input[0]: [16, 128, 8, 8] | output: [16, 128, 8, 8]
[encoder.2.2] -> input[0]: [16, 128, 8, 8] | output: [16, 128, 8, 8]
[encoder.3.0] -> input[0]: [16, 128, 8, 8] | output: [16, 256, 4, 4]
[encoder.3.1] -> input[0]: [16, 256, 4, 4] | output: [16, 256, 4, 4]
[encoder.3.2] -> input[0]: [16, 256, 4, 4] | output: [16, 256, 4, 4]
[encoder.4.0] -> input[0]: [16, 256

## TestVAE

In [9]:
import torch
import unittest
from models import VanillaVAE
from torchsummary import summary


class TestVAE(unittest.TestCase):

    def setUp(self) -> None:
        # self.model2 = VAE(3, 10)
        self.model = VanillaVAE(3, 10)

    def test_summary(self):
        print(summary(self.model, (3, 64, 64), device='cpu'))
        # print(summary(self.model2, (3, 64, 64), device='cpu'))

    def test_forward(self):
        x = torch.randn(16, 3, 64, 64)
        y = self.model(x)
        print("Model Output size:", y[0].size())
        # print("Model2 Output size:", self.model2(x)[0].size())

    def test_loss(self):
        x = torch.randn(16, 3, 64, 64)

        result = self.model(x)
        loss = self.model.loss_function(*result, M_N = 0.005)
        print(loss)

unittest.main(defaultTest='TestVAE', argv=['first-arg-is-ignored'], exit=False)

...
----------------------------------------------------------------------
Ran 3 tests in 0.191s

OK


Model Output size: torch.Size([16, 3, 64, 64])
{'loss': tensor(1.1410, grad_fn=<AddBackward0>), 'Reconstruction_Loss': tensor(1.1338), 'KLD': tensor(-1.4408)}
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 32, 32]             896
       BatchNorm2d-2           [-1, 32, 32, 32]              64
         LeakyReLU-3           [-1, 32, 32, 32]               0
            Conv2d-4           [-1, 64, 16, 16]          18,496
       BatchNorm2d-5           [-1, 64, 16, 16]             128
         LeakyReLU-6           [-1, 64, 16, 16]               0
            Conv2d-7            [-1, 128, 8, 8]          73,856
       BatchNorm2d-8            [-1, 128, 8, 8]             256
         LeakyReLU-9            [-1, 128, 8, 8]               0
           Conv2d-10            [-1, 256, 4, 4]         295,168
      BatchNorm2d-11            [-1, 256, 4, 4]             512
        

## TestVQVAE

In [1]:
import torch
import unittest
from models import VQVAE
# from torchsummary import summary
from torchinfo import summary
from tools import register_hooks


class TestVQVAE(unittest.TestCase):

    def setUp(self) -> None:
        # self.model2 = VAE(3, 10)
        self.model = VQVAE(3, 64, 512)

    def test_summary(self):
        print(summary(self.model, (1, 3, 64, 64), device='cpu'))
        # print(summary(self.model2, (3, 64, 64), device='cpu'))

    def test_forward(self):
        # register_hooks(self.model)
        print(sum(p.numel() for p in self.model.parameters() if p.requires_grad))
        x = torch.randn(16, 3, 64, 64)
        y = self.model(x)
        print("Model Output size:", y[0].size())
        # print("Model2 Output size:", self.model2(x)[0].size())

    def test_loss(self):
        x = torch.randn(16, 3, 64, 64)

        result = self.model(x)
        loss = self.model.loss_function(*result, M_N = 0.005)
        print(loss)

    def test_sample(self):
        self.model.cpu()
        y = self.model.sample(8, 'cpu')
        print(y.shape)

    def test_generate(self):
        x = torch.randn(16, 3, 64, 64)
        y = self.model.generate(x)
        print(y.shape)


unittest.main(defaultTest='TestVQVAE', argv=['first-arg-is-ignored'], exit=False)

9712707


.

Model Output size: torch.Size([16, 3, 64, 64])


.

torch.Size([16, 3, 64, 64])


...
----------------------------------------------------------------------
Ran 5 tests in 1.673s

OK


{'loss': tensor(1.0023, grad_fn=<AddBackward0>), 'Reconstruction_Loss': tensor(1.0007, grad_fn=<MseLossBackward0>), 'VQ_Loss': tensor(0.0016, grad_fn=<AddBackward0>)}
torch.Size([8, 3, 64, 64])
Layer (type:depth-idx)                   Output Shape              Param #
VQVAE                                    [1, 3, 64, 64]            --
├─Sequential: 1-1                        [1, 64, 16, 16]           --
│    └─Sequential: 2-1                   [1, 128, 32, 32]          --
│    │    └─Conv2d: 3-1                  [1, 128, 32, 32]          6,272
│    │    └─LeakyReLU: 3-2               [1, 128, 32, 32]          --
│    └─Sequential: 2-2                   [1, 256, 16, 16]          --
│    │    └─Conv2d: 3-3                  [1, 256, 16, 16]          524,544
│    │    └─LeakyReLU: 3-4               [1, 256, 16, 16]          --
│    └─Sequential: 2-3                   [1, 256, 16, 16]          --
│    │    └─Conv2d: 3-5                  [1, 256, 16, 16]          590,080
│    │    └─LeakyR